Data Preprocessing and Aggregation

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import timedelta, datetime

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Change working directory to project folder
try:
    os.chdir("/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence")
    print("Directory changed")
except OSError:
    print("Error: Can't change the Current Working Directory")

Mounted at /content/drive
Directory changed


In [3]:
df_original = pd.read_csv('data/food_waste_augmented_cl_updated.csv')
print(f"Original data shape: {df_original.shape}")

Original data shape: (119616, 29)


In [4]:
df_original.head()

,Date,Meal,Canteen_Section,Food_Category,Waste_Weight_kg,Unit_Price_per_kg,Is_Event_Day,Is_Holiday,Day_Type,Year,...,Month_Name,Weekday_Type,Cost_Loss,Waste_per_Price,Log_Waste,Foot_Traffic,Is_Waste_Outlier,Is_Cost_Outlier,Is_High_Waste,Is_High_Cost
0,2015-01-01 07:30:00,Breakfast,B,Meat,0.020,8.0,0,1,holiday,2015,...,January,Weekday,0.16,0.002500,-3.912023,1.80,0,0,0,0
1,2015-01-01 07:00:00,Breakfast,A,Soup,0.090,1.5,0,1,holiday,2015,...,January,Weekday,0.14,0.060000,-2.407946,1.20,0,0,0,0
2,2015-01-01 06:00:00,Breakfast,D,Rice,0.010,2.0,0,1,holiday,2015,...,January,Weekday,0.02,0.005000,-4.605170,0.56,0,0,0,0
3,2015-01-01 06:30:00,Breakfast,C,Meat,0.024,8.0,0,1,holiday,2015,...,January,Weekday,0.19,0.003000,-3.729701,0.06,0,0,0,0
4,2015-01-01 08:00:00,Breakfast,A,Meat,0.007,8.0,0,1,holiday,2015,...,January,Weekday,0.06,0.000875,-4.961845,0.07,0,0,0,0


## 1. Parse Timestamps & Create 30-Minute Bins

In [5]:
df = df_original.copy()

# Parse Date column to datetime
df['datetime'] = pd.to_datetime(df['Date'])

# Floor to nearest 30-minute bin
df['time_bin'] = df['datetime'].dt.floor('30min')

print(f"Date range: {df['time_bin'].min()} → {df['time_bin'].max()}")
print(f"Sample time bins:\n{df[['Date', 'datetime', 'time_bin']].head(10)}")

Date range: 2015-01-01 06:00:00 → 2025-08-10 19:30:00
Sample time bins:
                  Date            datetime            time_bin
0  2015-01-01 07:30:00 2015-01-01 07:30:00 2015-01-01 07:30:00
1  2015-01-01 07:00:00 2015-01-01 07:00:00 2015-01-01 07:00:00
2  2015-01-01 06:00:00 2015-01-01 06:00:00 2015-01-01 06:00:00
3  2015-01-01 06:30:00 2015-01-01 06:30:00 2015-01-01 06:30:00
4  2015-01-01 08:00:00 2015-01-01 08:00:00 2015-01-01 08:00:00
5  2015-01-01 08:30:00 2015-01-01 08:30:00 2015-01-01 08:30:00
6  2015-01-01 11:00:00 2015-01-01 11:00:00 2015-01-01 11:00:00
7  2015-01-01 11:30:00 2015-01-01 11:30:00 2015-01-01 11:30:00
8  2015-01-01 12:00:00 2015-01-01 12:00:00 2015-01-01 12:00:00
9  2015-01-01 12:30:00 2015-01-01 12:30:00 2015-01-01 12:30:00


## 2. Aggregate per (Canteen Section, Time Bin)

In [6]:
agg_dict = {
    'Waste_Weight_kg': 'sum',   # total waste per bin
    'Cost_Loss': 'sum',         # total cost loss per bin
    'Foot_Traffic': 'mean'      # average foot traffic per bin
}

df_agg = df.groupby(['Canteen_Section', 'time_bin']).agg(agg_dict).reset_index()

print(f"Aggregated shape: {df_agg.shape}")
print(f"Sections: {df_agg['Canteen_Section'].unique()}")
df_agg.head()

Aggregated shape: (99700, 5)
Sections: ['A' 'B' 'C' 'D']


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic
0,A,2015-01-01 07:00:00,0.090,0.14,1.20
1,A,2015-01-01 08:00:00,0.007,0.06,0.07
2,A,2015-01-01 08:30:00,0.026,0.05,0.07
3,A,2015-01-01 11:30:00,0.074,0.22,0.07
4,A,2015-01-01 13:30:00,0.036,0.11,0.07


## 3. Remove Zero‑Waste Bins (No Artificial Filling)

In [7]:
# Remove rows where no waste was recorded (Waste_Weight_kg == 0).
# This eliminates the large number of zero‑value entries that were previously
# added by the full reindexing step, keeping only bins that actually contain waste.
rows_before = len(df_agg)
df_agg = df_agg[df_agg['Waste_Weight_kg'] > 0].reset_index(drop=True)
rows_after = len(df_agg)
print(f"Rows before filter: {rows_before:,}")
print(f"Rows after filter : {rows_after:,} ({rows_before - rows_after:,} removed)")

# Note: The dataset now contains only time bins where waste was generated.
# There will be gaps in the time series, which the later feature engineering
# step (lags, rolling windows) will handle by shifting over the irregular sequence.
# This is intentional to reduce dataset size and avoid storing meaningless zeros.

df_agg.head()

Rows before filter: 99,700
Rows after filter : 99,700 (0 removed)


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic
0,A,2015-01-01 07:00:00,0.090,0.14,1.20
1,A,2015-01-01 08:00:00,0.007,0.06,0.07
2,A,2015-01-01 08:30:00,0.026,0.05,0.07
3,A,2015-01-01 11:30:00,0.074,0.22,0.07
4,A,2015-01-01 13:30:00,0.036,0.11,0.07


## 4. Add Time-Based & Cyclical Features

In [8]:
tb = df_agg['time_bin']

# Raw time features
df_agg['hour']        = tb.dt.hour
df_agg['minute']      = tb.dt.minute
df_agg['weekday']     = tb.dt.dayofweek          # 0=Monday, 6=Sunday
df_agg['month']       = tb.dt.month
df_agg['day_of_year'] = tb.dt.dayofyear
df_agg['quarter']     = tb.dt.quarter
df_agg['is_weekend']  = (tb.dt.dayofweek >= 5).astype(int)

# Cyclical encoding
df_agg['sin_hour']    = np.sin(2 * np.pi * df_agg['hour']    / 24)
df_agg['cos_hour']    = np.cos(2 * np.pi * df_agg['hour']    / 24)
df_agg['sin_minute']  = np.sin(2 * np.pi * df_agg['minute']  / 60)
df_agg['cos_minute']  = np.cos(2 * np.pi * df_agg['minute']  / 60)
df_agg['sin_weekday'] = np.sin(2 * np.pi * df_agg['weekday'] / 7)
df_agg['cos_weekday'] = np.cos(2 * np.pi * df_agg['weekday'] / 7)
df_agg['sin_month']   = np.sin(2 * np.pi * df_agg['month']   / 12)
df_agg['cos_month']   = np.cos(2 * np.pi * df_agg['month']   / 12)

print(f"Final shape: {df_agg.shape}")
print(f"Columns: {list(df_agg.columns)}")
df_agg.head()

Final shape: (99700, 20)
Columns: ['Canteen_Section', 'time_bin', 'Waste_Weight_kg', 'Cost_Loss', 'Foot_Traffic', 'hour', 'minute', 'weekday', 'month', 'day_of_year', 'quarter', 'is_weekend', 'sin_hour', 'cos_hour', 'sin_minute', 'cos_minute', 'sin_weekday', 'cos_weekday', 'sin_month', 'cos_month']


,Canteen_Section,time_bin,Waste_Weight_kg,Cost_Loss,Foot_Traffic,hour,minute,weekday,month,day_of_year,quarter,is_weekend,sin_hour,cos_hour,sin_minute,cos_minute,sin_weekday,cos_weekday,sin_month,cos_month
0,A,2015-01-01 07:00:00,0.090,0.14,1.20,7,0,3,1,1,1,0,0.965926,-0.258819,0.000000e+00,1.0,0.433884,-0.900969,0.5,0.866025
1,A,2015-01-01 08:00:00,0.007,0.06,0.07,8,0,3,1,1,1,0,0.866025,-0.500000,0.000000e+00,1.0,0.433884,-0.900969,0.5,0.866025
2,A,2015-01-01 08:30:00,0.026,0.05,0.07,8,30,3,1,1,1,0,0.866025,-0.500000,5.665539e-16,-1.0,0.433884,-0.900969,0.5,0.866025
3,A,2015-01-01 11:30:00,0.074,0.22,0.07,11,30,3,1,1,1,0,0.258819,-0.965926,5.665539e-16,-1.0,0.433884,-0.900969,0.5,0.866025
4,A,2015-01-01 13:30:00,0.036,0.11,0.07,13,30,3,1,1,1,0,-0.258819,-0.965926,5.665539e-16,-1.0,0.433884,-0.900969,0.5,0.866025


## 5. Save Preprocessed Data

In [9]:
output_path = 'data/food_waste_preprocessed.csv'
df_agg.to_csv(output_path, index=False)
print(f"Saved to {output_path}  |  shape: {df_agg.shape}")

Saved to data/food_waste_preprocessed.csv  |  shape: (99700, 20)
